# Real-estate analysis with Spark DataFrames and SQL

Cleaned portfolio version of an academic team project. The notebook uses an explicit schema and Spark SQL to analyse approximately 80,000 transactions in Toulouse and surrounding municipalities.

> Analytical note: the original course project used `price / (LivingArea + LandArea)` as a simplified unit-price indicator. A production analysis should validate separate living-area and land-area metrics.


# **SETTING**

In [ ]:
# Install PySpark once if it is not already available.
# %pip install -q pyspark==3.5.1

from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    DateType, DoubleType, IntegerType, StringType, StructField, StructType
)

spark = SparkSession.builder.appName(
    "Toulouse real-estate analysis"
).getOrCreate()

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")


**IMPORT SPARK**

In [ ]:
print(spark.version)


# **Analysis**



In [ ]:
sales_schema = StructType([
    StructField("SaleDate", DateType(), True),
    StructField("SaleType", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("postal_code", IntegerType(), True),
    StructField("City", StringType(), True),
    StructField("NbRooms", IntegerType(), True),
    StructField("NbLots", IntegerType(), True),
    StructField("ResidenceType", StringType(), True),
    StructField("LivingArea", IntegerType(), True),
    StructField("LandArea", IntegerType(), True),
])


In [ ]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

sales_df = spark.read.csv(
    str(DATA_DIR / "project_data_real_estate_toulouse_and_suburbs.csv"),
    header=True,
    sep=";",
    dateFormat="dd/MM/yyyy",
    schema=sales_schema,
)

sales_df.show(5)


In [ ]:
# register the sales_df DataFrame as a temporary view with the name 'sales' for further spark sql queries

sales_df.createOrReplaceTempView('sales')

# **Most sales**

In [ ]:
#1 The city with the highest sales volume when the sales type is sale and the residence type is house.
total_number_sales1_df = spark.sql(" SELECT City, count(*) as total_sales \
                                    FROM sales\
                                    WHERE SaleType = 'SALE' and ResidenceType = 'HOUSE'\
                                    GROUP BY City\
                                    ORDER BY total_sales desc")

total_number_sales1_df.show()

In [ ]:
#2 The city with the highest sales volume when the sales type is sale and the residence type is Apartment.
total_number_sales2_df = spark.sql(" SELECT City, count(*) as total_sales \
                                    FROM sales\
                                    WHERE SaleType = 'SALE' and ResidenceType = 'APARTMENT'\
                                    GROUP BY City\
                                    ORDER BY total_sales desc")


total_number_sales2_df.show()

In [ ]:
#3 The city with the highest sales volume when the sales type is SALE BEFORE COMPLETION and the residence type is Apartment.
total_number_sales3_df = spark.sql(" SELECT City, count(*) as total_sales \
                                    FROM sales\
                                    WHERE SaleType = 'SALE BEFORE COMPLETION' and ResidenceType = 'APARTMENT'\
                                    GROUP BY City\
                                    ORDER BY total_sales desc")


total_number_sales3_df.show()

In [ ]:
#4 The city with the highest sales volume when the sales type is SALE BEFORE COMPLETION and the residence type is House.
total_number_sales4_df = spark.sql(" SELECT City, count(*) as total_sales \
                                    FROM sales\
                                    WHERE SaleType = 'SALE BEFORE COMPLETION' and ResidenceType = 'HOUSE'\
                                    GROUP BY City\
                                    ORDER BY total_sales desc")


total_number_sales4_df.show()

Summary ： The cities with the highest number of sales by type of sale and by type of house are TOULOUSE

# **The evolution of the prices**



In [ ]:
#1 The unit_price of TOULOUSE 
price_per_square_meter_tls_df = spark.sql(" SELECT *, price/(LivingArea+LandArea) as unit_price \
                                    FROM sales\
                                    WHERE City='TOULOUSE'\
                                    ORDER BY SaleDate asc")


price_per_square_meter_tls_df.show(50)

In [ ]:
#2 The unit_price of the suburbs
price_per_square_meter_sub_df = spark.sql(" SELECT *, price/(LivingArea+LandArea) as unit_price \
                                    FROM sales\
                                    WHERE City !='TOULOUSE'\
                                    ORDER BY SaleDate asc")
price_per_square_meter_sub_df.show(50)

# **Differences in unit prices in the Toulouse region under the different postal codes**

In [ ]:
# Due to the results of the city with most sales
# we chose to analyse the unit price under different postal codes in Toulouse

unit_price_df = spark.sql(" SELECT *, price/(LivingArea+LandArea) as unit_price \
                                    FROM sales\
                                    WHERE City ='TOULOUSE'\
                                    ORDER BY unit_price desc")
unit_price_df.show(50)

# **Differences between apartment and houses**

In [ ]:
# 
unit_price_APT_df = spark.sql(" SELECT *, price/(LivingArea+LandArea) as unit_price \
                                    FROM sales\
                                    WHERE ResidenceType = 'APARTMENT'\
                                    ORDER BY unit_price desc")


unit_price_APT_df.show()

In [ ]:
unit_price_HOUSE_df = spark.sql(" SELECT *, price/(LivingArea+LandArea) as unit_price \
                                    FROM sales\
                                    WHERE ResidenceType = 'HOUSE'\
                                    ORDER BY unit_price desc")
unit_price_HOUSE_df.show()

# **Differences between types of sales**

In [ ]:
unit_price_sale_df = spark.sql(" SELECT *, price/(LivingArea+LandArea) as unit_price \
                                    FROM sales\
                                    WHERE SaleType = 'SALE'\
                                    ORDER BY unit_price desc")


unit_price_sale_df.show()

In [ ]:
unit_price_precompletion_df = spark.sql(" SELECT *, price/(LivingArea+LandArea) as unit_price \
                                    FROM sales\
                                    WHERE SaleType = 'SALE BEFORE COMPLETION'\
                                    ORDER BY unit_price desc")


unit_price_precompletion_df.show()